In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import cv2
import numpy as np
import os
import pandas as pd
from google.colab.patches import cv2_imshow
from skimage.feature import graycomatrix, graycoprops

def ekstraksi_lengkap_tanaman(folder_path):
    # List untuk menampung semua fitur
    data_fitur = []

    # Ambil daftar file gambar
    valid_formats = ('.jpg', '.jpeg', '.png', '.JPG', '.PNG')
    file_list = [f for f in os.listdir(folder_path) if f.lower().endswith(valid_formats)]

    if not file_list:
        print("Folder kosong atau path salah. Pastikan path mengarah ke folder gambar.")
        return None

    print(f"Memproses {len(file_list)} citra tanaman cabai...\n")

    for filename in file_list:
        path = os.path.join(folder_path, filename)
        img = cv2.imread(path)

        if img is not None:
            # --- PREPROCESSING ---
            # Resize agar seragam (misal 256x256)
            img = cv2.resize(img, (256, 256))

            # --- 1. EKSTRAKSI FITUR WARNA (HSV) ---
            hsv_img = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
            # Menghitung rata-rata (Mean) untuk H, S, V
            mean_hsv = cv2.mean(hsv_img)[:3]

            # --- 2. EKSTRAKSI FITUR TEKSTUR (GLCM) ---
            # GLCM membutuhkan gambar grayscale
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

            # Membuat Matrix GLCM (Jarak=1, Sudut=0 derajat)
            glcm = graycomatrix(gray, distances=[1], angles=[0], levels=256,
                                symmetric=True, normed=True)

            # Mengambil properti tekstur
            contrast = graycoprops(glcm, 'contrast')[0, 0]      # Kontras (Kekasaran)
            homogeneity = graycoprops(glcm, 'homogeneity')[0, 0] # Kemulusan
            energy = graycoprops(glcm, 'energy')[0, 0]          # Keteraturan pola
            correlation = graycoprops(glcm, 'correlation')[0, 0] # Hubungan piksel

            # --- SIMPAN DATA ---
            data_fitur.append({
                "File": filename,
                "Mean_Hue": round(mean_hsv[0], 2),    # Warna (Hijau/Kuning)
                "Mean_Sat": round(mean_hsv[1], 2),    # Kepekatan Warna
                "Mean_Val": round(mean_hsv[2], 2),    # Kecerahan
                "Contrast": round(contrast, 2),       # Tekstur Kasar
                "Homogeneity": round(homogeneity, 4), # Tekstur Halus
                "Energy": round(energy, 4),           # Konsistensi Warna
                "Correlation": round(correlation, 4)  # Pola Linier
            })
        else:
            print(f"Gagal memuat: {filename}")

    # Konversi ke DataFrame Pandas agar rapi
    df = pd.DataFrame(data_fitur)
    return df

PATH_DATASET = '/content/drive/MyDrive/Colab Notebooks/Dataset_Tanaman_cabai/'

df_hasil = ekstraksi_lengkap_tanaman(PATH_DATASET)

if df_hasil is not None:
    print("Hasil Ekstraksi Fitur Tanaman Cabai:")
    display(df_hasil.head(10))


Memproses 10 citra tanaman cabai...

Hasil Ekstraksi Fitur Tanaman Cabai:


,File,Mean_Hue,Mean_Sat,Mean_Val,Contrast,Homogeneity,Energy,Correlation
0,Cabai1.JPG,81.84,56.79,101.82,168.09,0.4577,0.0719,0.9804
1,Cabai6.JPG,80.63,38.24,123.98,126.92,0.5073,0.1073,0.9899
2,Cabai5.JPG,81.14,36.65,132.14,135.17,0.4666,0.0865,0.9881
3,Cabai4.JPG,86.32,35.76,117.42,109.46,0.4813,0.0789,0.9903
4,Cabai3.JPG,67.59,41.51,102.88,151.13,0.4738,0.0803,0.9850
5,Cabai2.JPG,75.29,45.78,98.27,117.31,0.4978,0.0894,0.9868
6,Cabai10.JPG,80.65,48.29,109.55,147.87,0.4629,0.0697,0.9839
7,Cabai9.JPG,77.53,48.77,98.25,157.70,0.4870,0.0722,0.9820
8,Cabai8.JPG,83.21,50.31,116.76,143.85,0.4496,0.0760,0.9824
9,Cabai7.JPG,88.28,46.44,110.11,129.80,0.4659,0.0788,0.9827
